# Data Ingestion & Tratamento de Dados

Este notebook faz a ingestão e o tratamento completo dos dados para o projeto Energy Poverty.

Todos os ficheiros de dados são lidos a partir da pasta `data/` neste directório.

**Capítulos:**
1. Household Income (INE API 2015–2021 + Projeção 2022–2026)
2. Consumo de Energia por Município (Carlota — `energy_household.csv`)
3. Preços de Energia Elétrica (Simão — `energy_prices.parquet`)
4. Eficiência Energética — Certificados SCE (Madalena)
5. Idade Mediana por Município
6. Gasto Energético Anual por CPE e Normalização de Nomes
7. Dataset Final — Índice de Pobreza Energética (`df_energyVSincome`)

## 0. Imports e configuração de paths

In [170]:
import requests
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

DATA_DIR = Path("data")

---
# 1. Household Income

Fonte: INE — Rendimento médio disponível por habitante (varcd=0009762)  
https://www.ine.pt/xportal/xmain?xpid=INE&xpgid=ine_indicadores&indOcorrCod=0009762&contexto=bd&selTab=tab2

## 1.1 Recolha de dados via API INE (2015–2021)

In [171]:
anos = range(2015, 2022)
income_rows = []

for ano in anos:
    url = (
        f"https://www.ine.pt/ine/json_indicador/pindica.jsp"
        f"?op=2&varcd=0009762&lang=PT&Dim1=S7A{ano}"
    )
    data = requests.get(url).json()

    for entry in data:
        # Cada entry tem uma chave "Dados" com os registos desse ano
        for ano_key, registos in entry.get("Dados", {}).items():
            for r in registos:
                geocod = r.get("geocod", "")
                # Só municípios têm geocod numérico — excluir regiões e agregados nacionais
                if not geocod or not geocod[0].isdigit():
                    continue
                income_rows.append({
                    "ANO":                    int(ano_key),
                    "Município":              r.get("geodsg"),
                    "Household Income (EUR)": pd.to_numeric(r.get("valor"), errors="coerce"),
                })

# Construir o DataFrame, remover duplicados e ordenar
df_household_income_raw = (
    pd.DataFrame(income_rows)
    .dropna(subset=["Household Income (EUR)"])
    .drop_duplicates()
    .sort_values(["ANO", "Município"])
    .reset_index(drop=True)
)

print(f"{len(df_household_income_raw)} registos | {df_household_income_raw['Município'].nunique()} municípios | Anos: {sorted(df_household_income_raw['ANO'].unique())}")
df_household_income_raw

2359 registos | 335 municípios | Anos: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]


,ANO,Município,Household Income (EUR)
0,2015,Abrantes,15108
1,2015,Aguiar da Beira,11334
2,2015,Alandroal,11515
3,2015,Albergaria-a-Velha,14776
4,2015,Albufeira,13363
...,...,...,...
2354,2021,Área Metropolitana de Lisboa,23321
2355,2021,Área Metropolitana do Porto,20110
2356,2021,Évora,22087
2357,2021,Ílhavo,20196


In [172]:
#  Média do rendimento por ano (anos com maior rendimento médio) 
df_household_income_raw.groupby("ANO")["Household Income (EUR)"].mean().round(2).sort_values(ascending=False).reset_index()

,ANO,Household Income (EUR)
0,2021,17253.19
1,2020,16533.68
2,2019,16317.03
3,2018,15741.34
4,2017,15102.47
5,2016,14683.76
6,2015,14164.22


In [173]:
#  Top 10 registos com maior rendimento (ANO + Município, overall) 
df_household_income_raw.nlargest(10, "Household Income (EUR)").reset_index(drop=True)

,ANO,Município,Household Income (EUR)
0,2021,Oeiras,29727
1,2021,Lisboa,29082
2,2020,Oeiras,28736
3,2019,Oeiras,28718
4,2019,Lisboa,28372
5,2018,Oeiras,28160
6,2020,Lisboa,28072
7,2018,Lisboa,27777
8,2017,Oeiras,27263
9,2021,Cascais,26785


In [174]:
#  Todos os registos ordenados por rendimento decrescente 
df_household_income_raw.sort_values("Household Income (EUR)", ascending=False).reset_index(drop=True)

,ANO,Município,Household Income (EUR)
0,2021,Oeiras,29727
1,2021,Lisboa,29082
2,2020,Oeiras,28736
3,2019,Oeiras,28718
4,2019,Lisboa,28372
...,...,...,...
2354,2015,Boticas,10547
2355,2015,Celorico de Basto,10335
2356,2016,Cinfães,10234
2357,2015,Resende,10183


## 1.2 Previsão de Household Income por município (até 2026)

**Objetivo**: Projetar valores de *household income* para 2022–2026, município a município.

O crescimento anual de cada município é calculado com base em três componentes com pesos fixos:
- **50%** – CAGR histórico do próprio município (crescimento pré-COVID, até 2019)
- **20%** – Fatores macroeconómicos: inflação e crescimento do PIB (10% cada)
- **30%** – Crescimento salarial: salário mínimo e salário médio (15% cada)

**Nota**: este notebook foi gerado em 2026-05-16.

### Fontes (para defender o trabalho)

Não é feito forecast da inflação/salário mínimo; são usadas séries publicadas.

- Salário mínimo Mensal nacional (tabela anual, inclui 2026): https://www.pordata.pt/portugal/evolucao+do+salario+minimo+nacional-74

- Salário Médio Líquido Mensal Em portugal - https://pt.tradingeconomics.com/portugal/wages

- Previsão macro (PIB) – PORDATA - https://www.pordata.pt/portugal/taxa+de+crescimento+do+pib-2298
    - Estimativa para 2026 https://www.bportugal.pt/publicacao/boletim-economico-marco-2026

-Inflação segundo Banco de Portugal - https://bpstat.bportugal.pt/serie/12704650
    - Estimativa para 2026: https://pt.tradingeconomics.com/portugal/inflation-cpi


- (Proxy para mediana) 'Rendimento mediano disponível' na PORDATA Retratos da Europa: https://retratos.europa.pordata.pt/custo-de-vida/portugal

### 1.2.0 Preparar o DataFrame para o pipeline de projeção

Partimos do `df_household_income_raw` já em memória e normalizamos as colunas para os nomes usados no restante pipeline.

In [175]:
# Partir do DataFrame já em memória, sem ler nenhum ficheiro intermédio
df_income_forecast = df_household_income_raw.copy()

# Normalizar nomes das colunas para os nomes usados no pipeline de projeção
rename_map = {
    'ANO': 'year',
    'Município': 'municipio',
    'Household Income (EUR)': 'income',
}
for k, v in rename_map.items():
    if k in df_income_forecast.columns:
        df_income_forecast = df_income_forecast.rename(columns={k: v})

# Conversão de tipos
df_income_forecast['year']      = df_income_forecast['year'].astype(int)
df_income_forecast['municipio'] = df_income_forecast['municipio'].astype(str)
df_income_forecast['income']    = pd.to_numeric(df_income_forecast['income'], errors='coerce')

# Ordenar por município e ano para os cálculos temporais seguintes
df_income_forecast = df_income_forecast.sort_values(['municipio', 'year']).reset_index(drop=True)

df_income_forecast.head(8)

,year,municipio,income
0,2015,Abrantes,15108
1,2016,Abrantes,15609
2,2017,Abrantes,16014
3,2018,Abrantes,16574
4,2019,Abrantes,17115
5,2020,Abrantes,17326
6,2021,Abrantes,18005
7,2015,Aguiar da Beira,11334


### 1.2.1 Calcular CAGR pré-COVID (até 2019)

CAGR significa Compound Annual Growth Rate. O objetivo é calcular o crescimento anual composto de cada município até 2019, evitando o efeito do choque COVID.

A lógica é fixar o município e medir a variação entre os pontos inicial e final do período pré-COVID. Por exemplo, se o income passa de 100 para 121 em 2 anos, o CAGR é (121/100)^(1/2) - 1 = 0,10, ou 10% ao ano.

In [176]:
def calculate_pre_covid_cagr(group: pd.DataFrame) -> float:
    """Calculate CAGR for one municipio using observations up to 2019."""
    """
    Args:
        group: DataFrame slice for a single municipio (all years for that municipio).
               The caller uses `df_income_forecast.groupby('municipio')`, so each `group` contains only one municipio,
               e.g. rows for 'Abrantes' from 2015..2019. This function will further filter to years <= 2019.

    Returns:
        CAGR as float (e.g. 0.10 for 10%/year) or np.nan if not computable.
    """
    # Filtra até 2019 para estimar crescimento estrutural sem o efeito COVID.
    g = group[group['year'] <= 2019].copy()

    # São necessários pelo menos dois pontos no tempo para calcular CAGR.
    # Exemplo: para Abrantes pode haver linhas para 2015,2016,2017,2018,2019 -> shape[0] >= 2 ok.
    if g.shape[0] < 2:
        return np.nan

    # Pegamos o primeiro e o último valor observados no período filtrado (mesmo municipio).
    start = g.iloc[0]['income']
    end = g.iloc[-1]['income']
    # Número de anos entre os dois pontos (p.ex. 2019 - 2015 = 4).
    years = g.iloc[-1]['year'] - g.iloc[0]['year']



    # Fórmula do CAGR: (end/start)^(1/years) - 1.
    # Exemplo: start=100, end=121, years=2 => (121/100)^(1/2) - 1 = 0.10 (10% ao ano).
    return (end / start) ** (1 / years) - 1

cagr_df = (
    df_income_forecast.groupby('municipio', as_index=False)
      .apply(lambda g: pd.Series({'cagr_pre_covid': calculate_pre_covid_cagr(g)}))
)

cagr_df.head()

/var/folders/f8/r14dd3cn66q9d1d3lmtz0lth0000gn/T/ipykernel_9068/3789747311.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({'cagr_pre_covid': calculate_pre_covid_cagr(g)}))


,municipio,cagr_pre_covid
0,Abrantes,0.031674
1,Aguiar da Beira,0.056820
2,Alandroal,0.048184
3,Albergaria-a-Velha,0.034377
4,Albufeira,0.034842


### 1.2.2 Tabela Macro com fatores económicos para extrapolar os Median Household Income (inflação, PIB, salários)

Dados com bases citadas acima.

- O ambiente do notebook pode não ter acesso à Internet, por isso as séries são definidas manualmente e validadas.
- É verificado que todos os anos necessários estão presentes para evitar buracos no modelo.

**Como preencher**: copiar os valores dos sites e substituir nos dicionários abaixo.

In [177]:
YEARS_FORECAST = [2022, 2023, 2024, 2025, 2026]

# Inflação anual (decimal). Fonte: INE / Banco de Portugal.
inflation = {2022: 0.0783, 2023: 0.0431, 2024: 0.0242, 2025: 0.0234, 2026: 0.0330}

# Crescimento real do PIB (decimal). Fonte: Banco de Portugal / OCDE.
gdp_growth = {2022: 0.070, 2023: 0.031, 2024: 0.022, 2025: 0.019, 2026: 0.018}

# Salário mínimo nacional (EUR/mês). Fonte: PORDATA.
min_wage = {2022: 705, 2023: 760, 2024: 820, 2025: 870, 2026: 920}

# Salário médio líquido (EUR/mês). Fonte: Trading Economics / PORDATA.
avg_wage = {2022: 1010, 2023: 1090, 2024: 1230, 2025: 1310, 2026: 1370}

# Construir tabela macro
macro = pd.DataFrame({
    'year': YEARS_FORECAST,
    'inflation': [inflation[y] for y in YEARS_FORECAST],
    'gdp_growth': [gdp_growth[y] for y in YEARS_FORECAST],
    'min_wage':   [min_wage[y]   for y in YEARS_FORECAST],
    'avg_wage':   [avg_wage[y]   for y in YEARS_FORECAST],
})

# Calcular taxas de crescimento anuais dos salários
macro['g_min_wage'] = macro['min_wage'].pct_change().fillna(0)
macro['g_avg_wage'] = macro['avg_wage'].pct_change().fillna(0)

macro

,year,inflation,gdp_growth,min_wage,avg_wage,g_min_wage,g_avg_wage
0,2022,0.0783,0.070,705,1010,0.000000,0.000000
1,2023,0.0431,0.031,760,1090,0.078014,0.079208
2,2024,0.0242,0.022,820,1230,0.078947,0.128440
3,2025,0.0234,0.019,870,1310,0.060976,0.065041
4,2026,0.0330,0.018,920,1370,0.057471,0.045802


### 1.2.3 Preparar base (último ano observado) e juntar CAGR

O ponto de partida para a projeção é o income de cada município em 2021 (último ano observado), ao qual se junta o CAGR calculado na secção anterior.

### 1.2.4 Loop de projeção (2022–2026)

Fórmula de crescimento anual com pesos fixos:

```
g_total = 0.50 * cagr_pre_covid
        + 0.10 * inflação
        + 0.10 * crescimento_pib
        + 0.15 * crescimento_salario_minimo
        + 0.15 * crescimento_salario_medio
```

A projeção aplica este `g_total` ao income do ano anterior, iterando de 2022 a 2026.

In [178]:
latest_year = int(df_income_forecast['year'].max())

base_df = df_income_forecast[df_income_forecast['year'] == latest_year].copy()
base_df = base_df.merge(cagr_df, on='municipio', how='left')

base_df.head()

,year,municipio,income,cagr_pre_covid
0,2021,Abrantes,18005,0.031674
1,2021,Aguiar da Beira,15269,0.056820
2,2021,Alandroal,14850,0.048184
3,2021,Albergaria-a-Velha,18064,0.034377
4,2021,Albufeira,15233,0.034842


In [179]:
projections = []
current_df = base_df.copy()

for _, row in macro.sort_values('year').iterrows():
    year = int(row['year'])

    next_df = current_df.copy()

    # Taxa de crescimento anual = média ponderada dos cinco fatores
    g_total = (
        0.50 * next_df['cagr_pre_covid'] +  # 50% tendência local pré-COVID
        0.10 * float(row['inflation'])    +  # 10% inflação
        0.10 * float(row['gdp_growth'])   +  # 10% crescimento do PIB
        0.15 * float(row['g_min_wage'])   +  # 15% crescimento do salário mínimo
        0.15 * float(row['g_avg_wage'])      # 15% crescimento do salário médio
    )

    next_df['income'] = next_df['income'] * (1 + g_total)
    next_df['year'] = year

    projections.append(next_df[['municipio', 'year', 'income']])
    current_df = next_df

projection_df = pd.concat(projections, ignore_index=True)

df_household_income_forecast = pd.concat(
    [df_income_forecast[['municipio', 'year', 'income']], projection_df],
    ignore_index=True
).sort_values(['municipio', 'year']).reset_index(drop=True)



df_household_income_forecast.head(20)

,municipio,year,income
0,Abrantes,2015,15108.000000
1,Abrantes,2016,15609.000000
2,Abrantes,2017,16014.000000
3,Abrantes,2018,16574.000000
4,Abrantes,2019,17115.000000
5,Abrantes,2020,17326.000000
6,Abrantes,2021,18005.000000
7,Abrantes,2022,18557.159252
8,Abrantes,2023,19426.196795
9,Abrantes,2024,20427.911658


### 1.2.5 Sanity checks rápidos

Top 10 municípios em 2026 e série temporal de Lisboa para validar a projeção.

In [180]:
# Top 10 municípios em 2026
df_household_income_forecast[df_household_income_forecast['year'] == 2026].sort_values('income', ascending=False).head(10)

,municipio,year,income
1655,Lisboa,2026,36168.432067
2279,Oeiras,2026,35695.231906
95,Alcochete,2026,32638.780358
899,Cascais,2026,31989.700650
2699,Porto,2026,31735.866756
1067,Coimbra,2026,30794.915006
1763,Mafra,2026,28487.374129
515,Arruda dos Vinhos,2026,28408.285461
3995,Área Metropolitana de Lisboa,2026,28107.810446
983,Castro Verde,2026,28034.719538


In [181]:
df_household_income_forecast[df_household_income_forecast['municipio'] == 'Lisboa'].sort_values('year')

,municipio,year,income
1644,Lisboa,2015,24337.000000
1645,Lisboa,2016,24794.000000
1646,Lisboa,2017,25169.000000
1647,Lisboa,2018,27777.000000
1648,Lisboa,2019,28372.000000
1649,Lisboa,2020,28072.000000
1650,Lisboa,2021,29082.000000
1651,Lisboa,2022,30081.782884
1652,Lisboa,2023,31602.157724
1653,Lisboa,2024,33349.005712


---
# 2. Consumo de Energia por Município (Carlota)

Fonte: ERSE — Consumos faturados por município e caracterização de PES (Pontos de Entrega de Serviço).  
Lemos diretamente o ficheiro `energy_household.csv` que já tem o consumo doméstico (Baixa Tensão) agregado por concelho e ano.

## 2.1 CPE and Energy Consumption — CSV Data Loading

### Normalize column names for easier use

In [182]:
df_energy_household = pd.read_csv(
    DATA_DIR / "energy_household.csv"
).drop(columns=["Unnamed: 0"], errors="ignore")

print(f"{len(df_energy_household)} registos | {df_energy_household['concelho'].nunique()} concelhos | Anos: {sorted(df_energy_household['ano'].unique())}")
df_energy_household

1112 registos | 278 concelhos | Anos: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,ano,distrito,concelho,energia_ativa_kwh,cpes,kwh_por_cpe
0,2022,Aveiro,Albergaria-a-Velha,5.060819e+07,14082,3593.821190
1,2022,Aveiro,Anadia,5.757067e+07,17134,3360.024863
2,2022,Aveiro,Arouca,4.187300e+07,12546,3337.557389
3,2022,Aveiro,Aveiro,1.793955e+08,50830,3529.322895
4,2022,Aveiro,Castelo de Paiva,2.649272e+07,7946,3334.094891
...,...,...,...,...,...,...
1107,2025,Évora,Reguengos de Monsaraz,2.303527e+07,7335,3140.459276
1108,2025,Évora,Vendas Novas,2.184646e+07,7092,3080.437505
1109,2025,Évora,Viana do Alentejo,1.221777e+07,3801,3214.356716
1110,2025,Évora,Vila Viçosa,1.476070e+07,4994,2955.686702


In [183]:
print(df_energy_household.isnull().sum())
df_energy_household.head()

ano                  0
distrito             0
concelho             0
energia_ativa_kwh    0
cpes                 0
kwh_por_cpe          0
dtype: int64


,ano,distrito,concelho,energia_ativa_kwh,cpes,kwh_por_cpe
0,2022,Aveiro,Albergaria-a-Velha,5.060819e+07,14082,3593.821190
1,2022,Aveiro,Anadia,5.757067e+07,17134,3360.024863
2,2022,Aveiro,Arouca,4.187300e+07,12546,3337.557389
3,2022,Aveiro,Aveiro,1.793955e+08,50830,3529.322895
4,2022,Aveiro,Castelo de Paiva,2.649272e+07,7946,3334.094891


---
# 3. Preços de Energia Elétrica (Simão)

Fonte: DGEG — Preços médios da eletricidade para uso doméstico, Banda DC (consumo anual ≤ 1000 kWh).  
Lemos o ficheiro `energy_prices.parquet` já processado que tem um preço por ano (€/kWh).

In [184]:
# O ficheiro energy_prices.parquet em data/ tem 2 entradas por ano.
# Usamos a média por ano para obter um único preço (€/kWh) por período.
df_energy_prices = (
    pd.read_parquet(DATA_DIR / "energy_prices.parquet")[["Ano", "Preco_Final"]]
    .groupby("Ano", as_index=False)["Preco_Final"]
    .mean()
)

print(f"{len(df_energy_prices)} registos | Anos: {sorted(df_energy_prices['Ano'].unique())}")
df_energy_prices

10 registos | Anos: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,Ano,Preco_Final
0,2016,0.232400
1,2017,0.225715
2,2018,0.226961
3,2019,0.216550
4,2020,0.212646
5,2021,0.212954
6,2022,0.221047
7,2023,0.218317
8,2024,0.252580
9,2025,0.241239


---
# 4. Energy Efficiency (Madalena)

In [185]:
df_eff_2015 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2015.csv', encoding='utf-16')
df_eff_2016 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2016.csv', encoding='utf-16')
df_eff_2017 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2017.csv', encoding='utf-16')
df_eff_2018 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2018.csv', encoding='utf-16')
df_eff_2019 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2019.csv', encoding='utf-16')
df_eff_2020 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2020.csv', encoding='utf-16')
df_eff_2021 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2021.csv', encoding='utf-16')
df_eff_2022 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2022.csv', encoding='utf-16')
df_eff_2023 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2023.csv', encoding='utf-16')
df_eff_2024Q1 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2024Q1.csv', encoding='utf-16')
df_eff_2024Q2 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2024Q2.csv', encoding='utf-16')
df_eff_2024Q3 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2024Q3.csv', encoding='utf-16')
df_eff_2024Q4 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2024Q4.csv', encoding='utf-16')
df_eff_2025Q1 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2025Q1.csv', encoding='utf-16')
df_eff_2025Q2 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2025Q2.csv', encoding='utf-16')
df_eff_2025Q3 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2025Q3.csv', encoding='utf-16')
df_eff_2025Q4 = pd.read_csv(DATA_DIR / 'EstatisticasSCE_2025Q4.csv', encoding='utf-16')

In [186]:
df_eff_2015.shape

(28173, 11)

In [187]:
df_eff_2016.shape

(29920, 11)

In [188]:
df_eff_2017.shape

(32642, 11)

In [189]:
df_eff_2018.shape

(34149, 11)

In [190]:
df_eff_2019.shape

(35698, 11)

In [191]:
df_eff_2020.shape

(35913, 11)

In [192]:
df_eff_2021.shape

(34855, 11)

In [193]:
df_eff_2022.shape

(37605, 11)

In [194]:
df_eff_2023.shape

(38682, 11)

In [195]:
df_eff_2023.shape

(38682, 11)

In [196]:
df_eff_2024Q1.shape

(9802, 11)

In [197]:
df_eff_2024Q2.shape

(9709, 11)

In [198]:
df_eff_2024Q3.shape

(9420, 11)

In [199]:
df_eff_2024Q4.shape

(9991, 11)

In [200]:
df_eff_2025Q1.shape

(9827, 11)

In [201]:
df_eff_2025Q2.shape

(9728, 11)

In [202]:
df_eff_2025Q3.shape

(9647, 11)

In [203]:
df_eff_2025Q4.shape

(9851, 11)

In [204]:
df_eff_2015.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [205]:
df_eff_2016.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [206]:
df_eff_2017.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [207]:
df_eff_2018.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [208]:
df_eff_2019.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [209]:
df_eff_2020.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [210]:
df_eff_2021.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [211]:
df_eff_2022.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [212]:
df_eff_2023.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [213]:
df_eff_2024Q1.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [214]:
df_eff_2024Q2.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [215]:
df_eff_2024Q3.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [216]:
df_eff_2024Q4.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [217]:
df_eff_2025Q1.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [218]:
df_eff_2025Q2.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [219]:
df_eff_2025Q3.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [220]:
df_eff_2025Q4.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [221]:
df_eff_2015_2025 = pd.concat([
    df_eff_2015, df_eff_2016, df_eff_2017, df_eff_2018, df_eff_2019,
    df_eff_2020, df_eff_2021, df_eff_2022, df_eff_2023,
    df_eff_2024Q1, df_eff_2024Q2, df_eff_2024Q3, df_eff_2024Q4,
    df_eff_2025Q1, df_eff_2025Q2, df_eff_2025Q3, df_eff_2025Q4
], ignore_index=True)

In [222]:
df_eff_2015_2025.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Emissões CO2', 'Área Média Útil de Pavimento',
       'Total de Certificados Emitidos'],
      dtype='object')

In [223]:
df_eff_2015_2025.shape

(385612, 11)

In [224]:
df_eff_2015_2025.head()

,Mês de Emissão,Região,NUTS III,Distrito,Concelho,Tipo de Doc / Contexto,Tipo de Edifício,Classe Energética,Total de Emissões CO2,Área Média Útil de Pavimento,Total de Certificados Emitidos
0,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,B,7.2000,119.500000,4
1,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,B-,7.8100,110.997500,4
2,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,C,31.8381,83.310833,12
3,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,D,45.6687,99.492222,9
4,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,F,19.6000,91.266666,3


In [225]:
df_eff_2015_2025.isnull().sum()

Mês de Emissão                    0
Região                            0
NUTS III                          0
Distrito                          0
Concelho                          0
Tipo de Doc / Contexto            0
Tipo de Edifício                  0
Classe Energética                 0
Total de Emissões CO2             0
Área Média Útil de Pavimento      0
Total de Certificados Emitidos    0
dtype: int64

In [226]:
df_eff_2015_2025['Tipo de Doc / Contexto'].unique()

array(['Edifícios Existentes', 'Projeto Edifícios Novos',
       'Projeto Renovação Edifícios', 'Edifícios Concluídos Novos',
       'Edifícios Concluídos Renovados'], dtype=object)

In [227]:
df_eff_2015_2025['Tipo de Edifício'].unique()

array(['Habitação', 'Serviços'], dtype=object)

We are going to eliminate values equal to "Serviços" in column "Tipo de Edifício". Energy poverty has to do with habitational buildings only.

In [228]:
df_eff_clean = df_eff_2015_2025.drop(df_eff_2015_2025[df_eff_2015_2025['Tipo de Edifício'] == 'Serviços'].index)

In [229]:
df_eff_clean.shape

(299110, 11)

In [230]:
df_eff_clean['Tipo de Edifício'].unique()

array(['Habitação'], dtype=object)

In [231]:
df_eff_clean['Classe Energética'].unique()

array(['B', 'B-', 'C', 'D', 'F', 'A', 'E', 'A+'], dtype=object)

We dropped coluns "Total de Emissões CO2" and "Área Média Útil de Pavimento", because those are not useful for energy poverty assessment.

In [232]:
df_eff_clean = df_eff_clean.drop(columns=['Total de Emissões CO2','Área Média Útil de Pavimento'])

In [233]:
df_eff_clean.columns

Index(['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho',
       'Tipo de Doc / Contexto', 'Tipo de Edifício', 'Classe Energética',
       'Total de Certificados Emitidos'],
      dtype='object')

In [234]:
df_eff_clean['Tipo de Doc / Contexto'].value_counts()

Tipo de Doc / Contexto
Edifícios Existentes              172020
Projeto Edifícios Novos            52067
Edifícios Concluídos Novos         34433
Projeto Renovação Edifícios        28621
Edifícios Concluídos Renovados     11969
Name: count, dtype: int64

A big part of the buildings are existing buildings, which means that, in our dataframe, we have certificates of new buildings and existing buildings.
We deleted the rows about the certificate of buildings still at the project stage because these buildings do not exist yet.

In [235]:
df_eff_clean = df_eff_clean.drop(df_eff_clean[df_eff_clean['Tipo de Doc / Contexto'] == 'Projeto Edifícios Novos'].index)
df_eff_clean = df_eff_clean.drop(df_eff_clean[df_eff_clean['Tipo de Doc / Contexto'] == 'Projeto Renovação Edifícios'].index)

In [236]:
df_eff_clean['Tipo de Doc / Contexto'].value_counts()

Tipo de Doc / Contexto
Edifícios Existentes              172020
Edifícios Concluídos Novos         34433
Edifícios Concluídos Renovados     11969
Name: count, dtype: int64

In [237]:
df_eff_clean.shape

(218422, 9)

In [238]:
df_eff_clean.head(20)

,Mês de Emissão,Região,NUTS III,Distrito,Concelho,Tipo de Doc / Contexto,Tipo de Edifício,Classe Energética,Total de Certificados Emitidos
0,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,B,4
1,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,B-,4
2,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,C,12
3,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,D,9
4,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,F,3
10,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,Habitação,E,5
12,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,ALBERGARIA-A-VELHA,Edifícios Existentes,Habitação,B-,3
13,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,ALBERGARIA-A-VELHA,Edifícios Existentes,Habitação,D,6
15,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,ALBERGARIA-A-VELHA,Edifícios Existentes,Habitação,E,6
17,2015-01,Portugal Continental,Baixo Vouga,AVEIRO,ANADIA,Edifícios Existentes,Habitação,A,3


In [239]:
df_eff_clean['Tipo de Edifício'].value_counts()

Tipo de Edifício
Habitação    218422
Name: count, dtype: int64

Now that all building types, are the same, we can erase this column.

In [240]:
df_eff_clean = df_eff_clean.drop(columns=['Tipo de Edifício'])

In [241]:
df_eff_clean["Mês de Emissão"].dtype

dtype('O')

We want the dataframe to be by year and not by month.

In [242]:
df_eff_clean["Mês de Emissão"] = pd.to_datetime(df_eff_clean["Mês de Emissão"])

In [243]:
df_eff_clean["Mês de Emissão"] = df_eff_clean["Mês de Emissão"].dt.year

In [244]:
df_energy_efficiency = (
    df_eff_clean.groupby(
        ['Mês de Emissão', 'Região', 'NUTS III', 'Distrito', 'Concelho', 'Tipo de Doc / Contexto', 'Classe Energética'],
        as_index=False
    )['Total de Certificados Emitidos'].sum()
)

In [245]:
df_eff_clean.head(100)

,Mês de Emissão,Região,NUTS III,Distrito,Concelho,Tipo de Doc / Contexto,Classe Energética,Total de Certificados Emitidos
0,2015,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,B,4
1,2015,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,B-,4
2,2015,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,C,12
3,2015,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,D,9
4,2015,Portugal Continental,Baixo Vouga,AVEIRO,AGUEDA,Edifícios Existentes,F,3
...,...,...,...,...,...,...,...,...
162,2015,Portugal Continental,Baixo Vouga,AVEIRO,VAGOS,Edifícios Existentes,C,1
163,2015,Portugal Continental,Baixo Vouga,AVEIRO,VAGOS,Edifícios Existentes,D,4
164,2015,Portugal Continental,Baixo Vouga,AVEIRO,VAGOS,Edifícios Existentes,F,1
168,2015,Portugal Continental,Baixo Vouga,AVEIRO,VAGOS,Edifícios Existentes,E,4


---
# 5. Median Age

In [246]:
df_median_age = pd.read_excel(
    DATA_DIR / "idade_mediana_por_municipio_1.xlsx"
)

In [247]:
df_median_age

,Concelho,Distrito,População (2023),Idade Mediana (2023)
0,Arcos de Valdevez,Viana do Castelo,20786,54.7
1,Caminha,Viana do Castelo,16324,50.8
2,Melgaço,Viana do Castelo,7607,60.6
3,Monção,Viana do Castelo,17974,54.0
4,Paredes de Coura,Viana do Castelo,8693,51.1
...,...,...,...,...
273,São Brás de Alportel,Faro,11551,47.4
274,Silves,Faro,39226,46.6
275,Tavira,Faro,27767,50.5
276,Vila do Bispo,Faro,5906,48.5


---
# 6. Gasto Energético Anual por CPE e Normalização de Nomes

Multiplicamos o consumo de energia (kWh/CPE) pelo preço da eletricidade (€/kWh) para obter o gasto energético anual por CPE.  
De seguida normalizamos os nomes dos concelhos para fazer o join com os dados de rendimento.

In [248]:
# Juntar o consumo com o preço da eletricidade para calcular o gasto anual por CPE
df_energy_expenditure = df_energy_household.merge(
    df_energy_prices[["Ano", "Preco_Final"]],
    left_on="ano",
    right_on="Ano",
    how="left"
).drop(columns=["Ano"])

df_energy_expenditure["annual_expenditure"] = (
    df_energy_expenditure["kwh_por_cpe"] * df_energy_expenditure["Preco_Final"]
)

print(f"NaN em annual_expenditure: {df_energy_expenditure['annual_expenditure'].isna().sum()}")
df_energy_expenditure.head()

NaN em annual_expenditure: 0


,ano,distrito,concelho,energia_ativa_kwh,cpes,kwh_por_cpe,Preco_Final,annual_expenditure
0,2022,Aveiro,Albergaria-a-Velha,5.060819e+07,14082,3593.821190,0.221047,794.404827
1,2022,Aveiro,Anadia,5.757067e+07,17134,3360.024863,0.221047,742.724757
2,2022,Aveiro,Arouca,4.187300e+07,12546,3337.557389,0.221047,737.758380
3,2022,Aveiro,Aveiro,1.793955e+08,50830,3529.322895,0.221047,780.147647
4,2022,Aveiro,Castelo de Paiva,2.649272e+07,7946,3334.094891,0.221047,736.993004


## 6.1 Normalização de Nomes

Para fazer o join entre os datasets, os nomes dos concelhos e municípios têm de estar no mesmo formato. A função `normalize` remove acentos, converte para minúsculas e elimina hífens e espaços extra.

In [249]:
def normalize(column):
    """Remove acentos, converte para minúsculas e elimina hífens e espaços extra."""
    normalized_values = []
    for x in column:
        x = str(x).strip().lower()
        x = unicodedata.normalize("NFKD", x)
        x = "".join(c for c in x if not unicodedata.combining(c))
        x = x.replace("-", " ")
        x = " ".join(x.split())
        normalized_values.append(x)
    return normalized_values

# Normalizar os nomes dos concelhos em energy_expenditure e os municípios no household income
df_energy_expenditure["concelho_limpo"] = normalize(df_energy_expenditure["concelho"])
df_household_income["municipio_limpo"]  = normalize(df_household_income["municipio"])

# Verificar que todos os concelhos têm correspondência no household income
energy_concelhos = set(df_energy_expenditure["concelho_limpo"].unique())
income_municipios = set(df_household_income["municipio_limpo"].unique())
sem_match = energy_concelhos - income_municipios
print(f"Concelhos sem correspondência no income: {len(sem_match)}")
if sem_match:
    print(sorted(sem_match))

Concelhos sem correspondência no income: 0


---
# 7. Dataset Final — Índice de Pobreza Energética

## 7.1 Merge Energia × Rendimento

Juntamos o gasto energético anual por CPE com o rendimento disponível por município para calcular o **Energy Expenditure Ratio** (EER):

```
EER = (Gasto Energético Anual / Rendimento Disponível) × 100
```

Se o EER for superior a 5%, o município/ano é classificado em **pobreza energética**.

In [250]:
# Fazer o join entre o gasto energético e o rendimento por município e ano
# A chave é concelho_limpo (energia) = municipio_limpo (income) e ano = year
df_energyVSincome = df_energy_expenditure.merge(
    df_household_income[["municipio_limpo", "year", "income"]],
    left_on=["concelho_limpo", "ano"],
    right_on=["municipio_limpo", "year"],
    how="left"
).drop(columns=["municipio_limpo", "year"])

print(f"Shape: {df_energyVSincome.shape}")
print(f"Municípios sem rendimento correspondente: {df_energyVSincome['income'].isna().sum()}")
df_energyVSincome.head()

Shape: (1116, 10)
Municípios sem rendimento correspondente: 0


,ano,distrito,concelho,energia_ativa_kwh,cpes,kwh_por_cpe,Preco_Final,annual_expenditure,concelho_limpo,income
0,2022,Aveiro,Albergaria-a-Velha,5.060819e+07,14082,3593.821190,0.221047,794.404827,albergaria a velha,19009.421736
1,2022,Aveiro,Anadia,5.757067e+07,17134,3360.024863,0.221047,742.724757,anadia,19007.067610
2,2022,Aveiro,Arouca,4.187300e+07,12546,3337.557389,0.221047,737.758380,arouca,16812.659110
3,2022,Aveiro,Aveiro,1.793955e+08,50830,3529.322895,0.221047,780.147647,aveiro,23232.854528
4,2022,Aveiro,Castelo de Paiva,2.649272e+07,7946,3334.094891,0.221047,736.993004,castelo de paiva,15419.346133


In [251]:
# Calcular o Energy Expenditure Ratio e o flag de pobreza energética
df_energyVSincome["energy_expenditure_ratio"] = (
    df_energyVSincome["annual_expenditure"] / df_energyVSincome["income"]
) * 100

# Classificar como pobreza energética se EER > 5%
df_energyVSincome["energy_poverty"] = (df_energyVSincome["energy_expenditure_ratio"] > 5).astype(int)

print(f"Municípios/anos em pobreza energética (EER > 5%): {df_energyVSincome['energy_poverty'].sum()}")
print(f"EER máximo: {df_energyVSincome['energy_expenditure_ratio'].max():.2f}%")
df_energyVSincome

Municípios/anos em pobreza energética (EER > 5%): 34
EER máximo: 6.25%


,ano,distrito,concelho,energia_ativa_kwh,cpes,kwh_por_cpe,Preco_Final,annual_expenditure,concelho_limpo,income,energy_expenditure_ratio,energy_poverty
0,2022,Aveiro,Albergaria-a-Velha,5.060819e+07,14082,3593.821190,0.221047,794.404827,albergaria a velha,19009.421736,4.179006,0
1,2022,Aveiro,Anadia,5.757067e+07,17134,3360.024863,0.221047,742.724757,anadia,19007.067610,3.907624,0
2,2022,Aveiro,Arouca,4.187300e+07,12546,3337.557389,0.221047,737.758380,arouca,16812.659110,4.388112,0
3,2022,Aveiro,Aveiro,1.793955e+08,50830,3529.322895,0.221047,780.147647,aveiro,23232.854528,3.357950,0
4,2022,Aveiro,Castelo de Paiva,2.649272e+07,7946,3334.094891,0.221047,736.993004,castelo de paiva,15419.346133,4.779664,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1111,2025,Évora,Reguengos de Monsaraz,2.303527e+07,7335,3140.459276,0.241239,757.601354,reguengos de monsaraz,24432.760223,3.100760,0
1112,2025,Évora,Vendas Novas,2.184646e+07,7092,3080.437505,0.241239,743.121760,vendas novas,25226.353668,2.945815,0
1113,2025,Évora,Viana do Alentejo,1.221777e+07,3801,3214.356716,0.241239,775.428301,viana do alentejo,23799.242438,3.258206,0
1114,2025,Évora,Vila Viçosa,1.476070e+07,4994,2955.686702,0.241239,713.026997,vila vicosa,24583.271702,2.900456,0


## 7.2 Municípios em Pobreza Energética

In [254]:
df_energyVSincome.shape

(1116, 12)

In [252]:
df_energyVSincome[df_energyVSincome["energy_poverty"] == 1][
    ["ano", "distrito", "concelho", "income", "annual_expenditure", "energy_expenditure_ratio"]
].sort_values("energy_expenditure_ratio", ascending=False)

,ano,distrito,concelho,income,annual_expenditure,energy_expenditure_ratio
174,2022,Porto,Paços de Ferreira,15814.759019,988.277570,6.249084
87,2022,Faro,Albufeira,16313.832927,986.481855,6.046904
645,2024,Faro,Albufeira,20586.063294,1183.899725,5.750977
95,2022,Faro,Loulé,18694.891245,1064.383104,5.693444
103,2022,Faro,Vila do Bispo,16617.929843,939.592003,5.654086
366,2023,Faro,Albufeira,18174.336852,1026.401369,5.647531
173,2022,Porto,Paredes,16095.511456,903.316845,5.612228
732,2024,Porto,Paços de Ferreira,20392.373639,1143.279919,5.606409
167,2022,Porto,Felgueiras,15899.457346,884.229548,5.561382
151,2022,Portalegre,Arronches,17023.339829,944.464496,5.548056


---
## 7.3 Sumário dos Datasets Disponíveis

Todos os datasets estão em memória e prontos para usar nas análises seguintes.

In [169]:
print("=== Datasets disponíveis ===")
print(f"df_household_income_raw  : {df_household_income_raw.shape}  — Household Income por município 2015–2021 (INE API)")
print(f"df_household_income      : {df_household_income.shape}  — Household Income com previsão 2015–2026")
print(f"df_energy_household      : {df_energy_household.shape}  — Consumo doméstico kWh/CPE por concelho/ano")
print(f"df_energy_prices         : {df_energy_prices.shape}  — Preços energia elétrica Banda DC por ano")
print(f"df_energy_efficiency     : {df_energy_efficiency.shape}  — Certificados energéticos por concelho/ano")
print(f"df_median_age            : {df_median_age.shape}  — Idade mediana por município 2023")
print(f"df_energy_expenditure    : {df_energy_expenditure.shape}  — Gasto energético anual por CPE")
print(f"df_energyVSincome        : {df_energyVSincome.shape}  — Dataset final com EER e flag de pobreza energética")

=== Datasets disponíveis ===
df_household_income_raw  : (2359, 3)  — Household Income por município 2015–2021 (INE API)
df_household_income      : (4044, 4)  — Household Income com previsão 2015–2026
df_energy_household      : (1112, 6)  — Consumo doméstico kWh/CPE por concelho/ano
df_energy_prices         : (10, 2)  — Preços energia elétrica Banda DC por ano
df_energy_efficiency     : (36199, 8)  — Certificados energéticos por concelho/ano
df_median_age            : (278, 4)  — Idade mediana por município 2023
df_energy_expenditure    : (1112, 9)  — Gasto energético anual por CPE
df_energyVSincome        : (1116, 12)  — Dataset final com EER e flag de pobreza energética
